# 005 — Secondary Metrics (Channel-Aware, ADTQC, Affiliation-Based)

The paper's evaluation is a five-level priority hierarchy (Table 3), and so
far we've only used level 1 — the corrected event-wise F-beta. This notebook
adds the rest, applied retroactively to the six models already built
(GlobalSTD3/5 from notebook 003, PCC/HBOS/iForest/KNN from notebook 004) —
no new models, just the full scoring picture on what we already have.

## Scope for each metric

- **Channel-aware F-score — GlobalSTD only.** The paper states this
  explicitly (Table 14): PCC/HBOS/iForest/KNN "only give global scores, so it
  is impossible to calculate subsystem-aware and channel-aware scores for
  them." GlobalSTD is the only algorithm we have with genuine per-channel
  predictions.
- **Subsystem-aware — not computed at all.** The paper doesn't report it for
  lightweight subsets, since they're drawn from a single subsystem by
  construction (channels 41-46 are all subsystem 4 for Mission1) — the metric
  would be a trivial passthrough of the channel-aware result.
- **ADTQC and affiliation-based — all six.** These work from a plain
  (timestamp, is_anomaly) series. For PCC/HBOS/iForest/KNN, which only ever
  produce one combined score, the same series gets duplicated across every
  target channel's key in the input dict — the most direct reading of how the
  paper could report these for algorithms it also says can't support
  channel-aware scoring.

## Implementation: same standard as before

- **Affiliation-based** uses the ESA-ADB authors' own patched fork of Huet et
  al.'s `affiliation-metrics-py`, installed directly from the specific
  subdirectory of the ESA-ADB repo — **not** the plain upstream package.
  Checked this carefully: the fork replaces `NaN` with `0.5` for
  empty-detection affiliation zones, exactly as Appendix C.2.3 describes
  ("empty detections get a precision of 0.5"). Confirmed the plain upstream
  package does **not** have this fix — on the same input, it silently
  produces a materially different number (`0.759` vs the correct `0.673` in
  one direct test). Installing from the specific patched subdirectory avoids
  that trap entirely.
- **Channel-aware F-score and ADTQC** are ported directly from
  `ranking_metrics.py` and `latency_metrics.py`.
- **Every one of the three metrics below is validated against the reference
  implementation's own worked example before touching real data** — same
  discipline as the event-wise scorer in notebook 003.

## A caveat worth stating plainly

The paper's Table 16/17 reference numbers used for comparison below were
manually transcribed from a PDF extraction that was noticeably messier for
these wide, many-column tables than Table 4 was (columns ran together, some
values got cut off during extraction). They're included as a useful
directional check, but **worth spot-checking against your own copy of the
paper** before treating any single digit as ground truth — unlike Table 4's
numbers, which came from cleaner text.


## 0. Imports

In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import portion as P
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "portion"])
    import portion as P

try:
    from affiliation.metrics import pr_from_events
except ImportError:
    import subprocess, sys
    # the SPECIFIC patched subdirectory, not the plain upstream package -- see the intro
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                    "git+https://github.com/kplabs-pl/ESA-ADB.git"
                    "#subdirectory=timeeval/metrics/affiliation_based_metrics_repo"])
    from affiliation.metrics import pr_from_events

print(f"pandas version: {pd.__version__}")


pandas version: 2.2.3


## 0.1 Mount Google Drive (same pattern as notebooks 002/003/004)

In [3]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path("/content/drive/MyDrive/BeaconProject")
    IN_COLAB = True
except ImportError:
    print("Not running in Colab — skipping Drive mount, falling back to a local path.")
    DRIVE_ROOT = Path("data/raw_drive_fallback")
    IN_COLAB = False

print(f"IN_COLAB = {IN_COLAB}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
IN_COLAB = True


## 1. Config

Table 16/17 numbers below cover the six algorithms we actually have. See the
intro's caveat on transcription quality for these specific tables.


In [4]:
MISSION_CONFIG = {
    "ESA-Mission1": {
        "lightweight_channels": [f"channel_{i}" for i in range(41, 47)],
        "test_data_split": "2007-01-01",
        "resampling_rule": pd.Timedelta(seconds=30),
        # Table 16 (Mission1 lightweight, excluding Communication Gap) -- transcribed, spot-check recommended
        "table16": {
            "PCC":     {"alarming_precision": 0.033, "adtqc_after_ratio": 0.833, "adtqc_score": 0.840,
                       "aff_precision": 0.535, "aff_recall": 0.334, "aff_f0.5": 0.477},
            "HBOS":    {"alarming_precision": 0.047, "adtqc_after_ratio": 0.763, "adtqc_score": 0.781,
                       "aff_precision": 0.543, "aff_recall": 0.352, "aff_f0.5": 0.490},
            "iForest": {"alarming_precision": 0.017, "adtqc_after_ratio": 0.711, "adtqc_score": 0.784,
                       "aff_precision": 0.543, "aff_recall": 0.357, "aff_f0.5": 0.492},
            "KNN":     {"alarming_precision": 0.017, "adtqc_after_ratio": 0.612, "adtqc_score": 0.803,
                       "aff_precision": 0.522, "aff_recall": 0.322, "aff_f0.5": 0.464},
            "GlobalSTD3": {"alarming_precision": 0.057, "adtqc_after_ratio": 0.929, "adtqc_score": 0.770,
                          "channel_precision": 0.431, "channel_recall": 0.285, "channel_f0.5": 0.351,
                          "aff_precision": 0.559, "aff_recall": 0.375, "aff_f0.5": 0.509},
            "GlobalSTD5": {"alarming_precision": 0.035, "adtqc_after_ratio": 0.909, "adtqc_score": 0.688,
                          "channel_precision": 0.169, "channel_recall": 0.159, "channel_f0.5": 0.167,
                          "aff_precision": 0.699, "aff_recall": 0.422, "aff_f0.5": 0.618},
        },
    },
    "ESA-Mission2": {
        "lightweight_channels": [f"channel_{i}" for i in range(18, 29)],
        "test_data_split": "2001-10-01",
        "resampling_rule": pd.Timedelta(seconds=18),
        # Table 17 (Mission2 lightweight, excluding Communication Gap) -- transcribed, spot-check recommended
        "table16": {
            "PCC":     {"alarming_precision": 0.061, "adtqc_after_ratio": 0.983, "adtqc_score": 0.999,
                       "aff_precision": 0.890, "aff_recall": 0.580, "aff_f0.5": 0.804},
            "HBOS":    {"alarming_precision": 0.105, "adtqc_after_ratio": 0.994, "adtqc_score": 0.990,
                       "aff_precision": 0.936, "aff_recall": 0.867, "aff_f0.5": 0.921},
            "iForest": {"alarming_precision": 0.075, "adtqc_after_ratio": 1.000, "adtqc_score": 0.991,
                       "aff_precision": 0.982, "aff_recall": 0.952, "aff_f0.5": 0.976},
            "KNN":     {"alarming_precision": 0.060, "adtqc_after_ratio": 0.391, "adtqc_score": 0.724,
                       "aff_precision": 0.561, "aff_recall": 0.243, "aff_f0.5": 0.445},
            "GlobalSTD3": {"alarming_precision": 0.054, "adtqc_after_ratio": 0.946, "adtqc_score": 0.997,
                          "channel_precision": 0.951, "channel_recall": 0.462, "channel_f0.5": 0.767,
                          "aff_precision": 0.740, "aff_recall": 0.296, "aff_f0.5": 0.612},
            "GlobalSTD5": {"alarming_precision": 0.061, "adtqc_after_ratio": 0.989, "adtqc_score": 0.997,
                          "channel_precision": 0.992, "channel_recall": 0.372, "channel_f0.5": 0.723,
                          "aff_precision": 0.935, "aff_recall": 0.717, "aff_f0.5": 0.881},
        },
    },
}

# --- EDIT THIS ONE LINE to switch missions ---
ACTIVE_MISSION = "ESA-Mission1"

CFG = MISSION_CONFIG[ACTIVE_MISSION]
TARGET_CHANNELS = CFG["lightweight_channels"]
RESAMPLING_RULE = CFG["resampling_rule"]

RAW_DATA_ROOT = DRIVE_ROOT / ACTIVE_MISSION / ACTIVE_MISSION
PREPROCESSED_DIR = DRIVE_ROOT / "preprocessed" / ACTIVE_MISSION
PREDICTIONS_PATH = DRIVE_ROOT / "predictions" / ACTIVE_MISSION / "unsupervised_baselines.csv"

BETA = 0.5

print(f"Active mission: {ACTIVE_MISSION}")
print(f"Target channels: {TARGET_CHANNELS}")


Active mission: ESA-Mission1
Target channels: ['channel_41', 'channel_42', 'channel_43', 'channel_44', 'channel_45', 'channel_46']


## 2. Load preprocessed data, raw labels, and notebook 004's cached predictions

In [5]:
anomaly_cols = [f"is_anomaly_{ch}" for ch in TARGET_CHANNELS]
dtypes = {ch: np.float64 for ch in TARGET_CHANNELS}
dtypes.update({c: np.uint8 for c in anomaly_cols})

train_df = pd.read_csv(PREPROCESSED_DIR / "train.csv", index_col="timestamp", parse_dates=True, dtype=dtypes)
test_df = pd.read_csv(PREPROCESSED_DIR / "test.csv", index_col="timestamp", parse_dates=True, dtype=dtypes)

labels_df = pd.read_csv(RAW_DATA_ROOT / "labels.csv", parse_dates=["StartTime", "EndTime"])
anomaly_types_df = pd.read_csv(RAW_DATA_ROOT / "anomaly_types.csv")
if labels_df["StartTime"].dt.tz is not None:
    labels_df["StartTime"] = labels_df["StartTime"].dt.tz_localize(None)
if labels_df["EndTime"].dt.tz is not None:
    labels_df["EndTime"] = labels_df["EndTime"].dt.tz_localize(None)
labels_df = labels_df.merge(anomaly_types_df[["ID", "Category"]], on="ID", how="left")

y_true_with_channel = labels_df[labels_df["Channel"].isin(TARGET_CHANNELS)].copy()
y_true_with_channel = y_true_with_channel[y_true_with_channel["StartTime"] >= pd.to_datetime(CFG["test_data_split"])]

predictions_raw = {}
_cached = pd.read_csv(PREDICTIONS_PATH, index_col="timestamp", parse_dates=True)
for col in _cached.columns:
    predictions_raw[col] = _cached[col].values
print(f"Loaded cached predictions from notebook 004: {list(predictions_raw.keys())}")
print(f"Total events in test period: {y_true_with_channel['ID'].nunique()}")


Loaded cached predictions from notebook 004: ['PCC', 'HBOS', 'iForest', 'KNN']
Total events in test period: 65


## 3. Recompute GlobalSTD

Fast (~seconds) and never saved to Drive, unlike the PyOD models — cheaper to
just recompute here than to add caching for something this quick. Same
validated logic as notebook 003; kept per-channel this time (not combined
via OR) since channel-aware scoring specifically needs that.


In [6]:
def fit_global_std(train_df, target_channels):
    means, stds = {}, {}
    for ch in target_channels:
        nominal_values = train_df.loc[train_df[f"is_anomaly_{ch}"] == 0, ch]
        means[ch] = nominal_values.mean()
        std = nominal_values.std()
        stds[ch] = std if std > 0 else 1.0
    return means, stds


def predict_global_std(df, target_channels, means, stds, tol):
    preds = {}
    for ch in target_channels:
        upper, lower = means[ch] + tol * stds[ch], means[ch] - tol * stds[ch]
        preds[ch] = ((df[ch] > upper) | (df[ch] < lower)).astype(np.uint8)
    return pd.DataFrame(preds, index=df.index)


train_means, train_stds = fit_global_std(train_df, TARGET_CHANNELS)
globalstd_preds = {
    "GlobalSTD3": predict_global_std(test_df, TARGET_CHANNELS, train_means, train_stds, 3.0),
    "GlobalSTD5": predict_global_std(test_df, TARGET_CHANNELS, train_means, train_stds, 5.0),
}
print("GlobalSTD3/5 recomputed.")


GlobalSTD3/5 recomputed.


## 4. Shared helper: `convert_time_series_to_events`

Same implementation used by all three metrics below and by notebook 003's
event-wise scorer — turns a (timestamp, binary) series into a set of
closed/closed-open intervals representing contiguous runs of detections.


In [7]:
def convert_time_series_to_events(vector) -> P.Interval:
    vector = np.asarray(vector)

    def find_runs(x):
        x = np.asanyarray(x)
        n = x.shape[0]
        if n == 0:
            return np.array([]), np.array([]), np.array([])
        loc_run_start = np.empty(n, dtype=bool)
        loc_run_start[0] = True
        np.not_equal(x[:-1], x[1:], out=loc_run_start[1:])
        run_starts = np.nonzero(loc_run_start)[0]
        run_values = x[loc_run_start]
        run_lengths = np.diff(np.append(run_starts, n))
        run_ends = run_starts + run_lengths
        return np.stack((run_starts[run_values > 0], run_ends[run_values > 0])).transpose()

    non_zero_runs = find_runs(vector[..., 1])
    events = []
    n = len(vector)
    for x, y in non_zero_runs:
        if y == n:
            events.append(P.closed(vector[..., 0][x], vector[..., 0][y - 1]))
        else:
            events.append(P.closedopen(vector[..., 0][x], vector[..., 0][y]))
    return P.Interval(*events)


## 5. Affiliation-based F-score

Ported from `ESA_ADB_metrics.py`'s affiliation-based scoring block, using the
patched fork installed above.


In [8]:
from collections import defaultdict

class AffiliationScorer:
    def __init__(self, betas=1.0, select_labels=None, full_range=None):
        self._betas = np.atleast_1d(betas)
        self.full_range = full_range
        if select_labels is None or len(select_labels) == 0:
            self.selected_labels = {}
        else:
            self.selected_labels = {c: np.atleast_1d(v) for c, v in select_labels.items()}

    def score(self, y_true: pd.DataFrame, y_pred) -> dict:
        y_pred = np.asarray(y_pred)

        if self.full_range is None:
            self.full_range = (min(y_true["StartTime"].min(), min(y_pred[..., 0])),
                               max(y_true["EndTime"].max(), max(y_pred[..., 0])))
        if y_pred[0, 0] > self.full_range[0]:
            y_pred = np.array([np.array([self.full_range[0], y_pred[0, 1]]), *y_pred])
        if y_pred[-1, 0] < self.full_range[1]:
            y_pred = np.array([*y_pred, np.array([self.full_range[1], y_pred[-1, 1]])])

        events_pred = convert_time_series_to_events(y_pred)
        events_gt = P.Interval(*[P.closed(*row) for _, row in y_true[["StartTime", "EndTime"]].iterrows()])

        events_pred_ns = [(e.lower.value, e.upper.value if e.lower.value != e.upper.value else e.upper.value + 1)
                          for e in events_pred]
        events_gt_ns = [(e.lower.value, e.upper.value if e.lower.value != e.upper.value else e.upper.value + 1)
                        for e in events_gt]

        corrected_full_upper_range = max(np.max(events_pred_ns), np.max(events_gt_ns), self.full_range[1].value)
        score_dict = pr_from_events(events_pred_ns, events_gt_ns,
                                    (self.full_range[0].value, corrected_full_upper_range))

        precision_dict = defaultdict(list)
        recall_dict = defaultdict(list)
        for pr, rec, zone in zip(score_dict["individual_precision_probabilities"],
                                 score_dict["individual_recall_probabilities"], events_gt):
            intersections = [P.closed(*row) & zone for _, row in y_true[["StartTime", "EndTime"]].iterrows()]
            y_true_in_zone = y_true[[not inter.empty for inter in intersections]]

            filtered_y_true_in_zone = y_true_in_zone.copy()
            for col, val in self.selected_labels.items():
                filtered_y_true_in_zone = filtered_y_true_in_zone[filtered_y_true_in_zone[col].isin(val)]

            if len(y_true_in_zone) > len(filtered_y_true_in_zone):
                continue

            for id_ in y_true_in_zone["ID"]:
                precision_dict[id_].append(pr)
                recall_dict[id_].append(rec)

        precision_list = [np.mean(pr) for pr in precision_dict.values()]
        recall_list = [np.mean(rec) for rec in recall_dict.values()]
        precision = np.mean(precision_list)
        recall = np.mean(recall_list)

        result = {"AFF_precision": precision, "AFF_recall": recall}
        for b in self._betas:
            result[f"AFF_F_{b:.2f}"] = ((1 + b ** 2) * precision * recall) / (b ** 2 * precision + recall)
        return result


# --- validate against the reference's own worked example ---
_full_range = (pd.to_datetime("2015-01-01"), pd.to_datetime("2015-01-15"))
_y_true = pd.DataFrame([
    ["id_0", pd.to_datetime("2015-01-01"), pd.to_datetime("2015-01-02")],
    ["id_1", pd.to_datetime("2015-01-04"), pd.to_datetime("2015-01-05")],
    ["id_2", pd.to_datetime("2015-01-07"), pd.to_datetime("2015-01-08")],
], columns=["ID", "StartTime", "EndTime"])
_y_pred = [[pd.to_datetime("2015-01-01"), 0], [pd.to_datetime("2015-01-04"), 1], [pd.to_datetime("2015-01-09"), 0]]

_result = AffiliationScorer(betas=0.5, full_range=_full_range).score(_y_true, _y_pred)
assert abs(_result["AFF_precision"] - 0.6728395061728394) < 1e-9
assert abs(_result["AFF_recall"] - 0.6666666666666666) < 1e-9
assert abs(_result["AFF_F_0.50"] - 0.6715958102279728) < 1e-9
print("PASS: AffiliationScorer matches the reference implementation exactly.")


PASS: AffiliationScorer matches the reference implementation exactly.


## 6. Channel-aware F-score

Ported from `ranking_metrics.py`'s `ChannelAwareFScore`. GlobalSTD only —
see the intro for why.


In [9]:
class ChannelAwareFScore:
    def __init__(self, beta: float = 0.5, select_labels=None, full_range=None):
        self._beta = beta
        self.full_range = full_range
        if select_labels is None or len(select_labels) == 0:
            self.selected_labels = {}
        else:
            self.selected_labels = {c: np.atleast_1d(v) for c, v in select_labels.items()}

    def _pr_re_f(self, tp, fp, fn):
        precision = 0.0 if (tp + fp) == 0 else tp / (tp + fp)
        recall = 0.0 if (tp + fn) == 0 else tp / (tp + fn)
        divider = (self._beta ** 2) * precision + recall
        f_score = 0.0 if divider == 0 else ((1 + self._beta ** 2) * precision * recall) / divider
        return precision, recall, f_score

    def score(self, y_true: pd.DataFrame, y_pred: dict) -> dict:
        all_channels = list(y_pred.keys())
        y_pred = {c: np.asarray(v) for c, v in y_pred.items()}

        min_ts = min(v[..., 0].min() for v in y_pred.values())
        max_ts = max(v[..., 0].max() for v in y_pred.values())
        if self.full_range is None:
            self.full_range = (min(y_true["StartTime"].min(), min_ts), max(y_true["EndTime"].max(), max_ts))
        for c in all_channels:
            if y_pred[c][0, 0] > self.full_range[0]:
                y_pred[c] = np.array([np.array([self.full_range[0], y_pred[c][0, 1]]), *y_pred[c]])
            if y_pred[c][-1, 0] < self.full_range[1]:
                y_pred[c] = np.array([*y_pred[c], np.array([self.full_range[1], y_pred[c][-1, 1]])])

        events_pred_per_channel = {c: convert_time_series_to_events(v) for c, v in y_pred.items()}

        filtered_y_true = y_true.copy()
        for col, val in self.selected_labels.items():
            filtered_y_true = filtered_y_true[filtered_y_true[col].isin(val)]

        point_anomalies = (filtered_y_true["StartTime"] == filtered_y_true["EndTime"])
        filtered_y_true = filtered_y_true.copy()
        filtered_y_true.loc[point_anomalies, "EndTime"] = (
            filtered_y_true.loc[point_anomalies, "StartTime"] + pd.Timedelta(milliseconds=1))

        unique_ids = filtered_y_true["ID"].unique()
        aid_channels_intervals = {}
        for aid in unique_ids:
            gt = filtered_y_true[filtered_y_true["ID"] == aid]
            channels_intervals = {}
            for c in all_channels:
                c_gt = gt[gt["Channel"] == c]
                channels_intervals[c] = P.Interval(*[P.closed(*row) for _, row in c_gt[["StartTime", "EndTime"]].iterrows()])
            aid_channels_intervals[aid] = channels_intervals

        precisions, recalls, f_scores = [], [], []
        for aid in unique_ids:
            channels_intervals = aid_channels_intervals[aid]
            full_interval = P.Interval(*list(channels_intervals.values()))

            tp = fp = fn = 0
            for c in all_channels:
                is_affected = not channels_intervals[c].empty
                detection_interval = full_interval & events_pred_per_channel[c]
                is_detected = not detection_interval.empty
                if is_affected and is_detected:
                    tp += 1
                elif is_affected and not is_detected:
                    fn += 1
                elif not is_affected and is_detected:
                    overlaps_other_event = False
                    for other_id, other_intervals in aid_channels_intervals.items():
                        if other_id == aid or other_intervals[c].empty:
                            continue
                        if not (detection_interval & other_intervals[c]).empty:
                            overlaps_other_event = True
                            break
                    if not overlaps_other_event:
                        fp += 1

            p, r, f = self._pr_re_f(tp, fp, fn)
            precisions.append(p)
            recalls.append(r)
            f_scores.append(f)

        return {
            "channel_precision": np.mean(precisions),
            "channel_recall": np.mean(recalls),
            f"channel_F{self._beta:.2f}": np.mean(f_scores),
        }


# --- validate against the reference's own worked example ---
_y_true_ch = pd.DataFrame([
    ["id_0", "ch1", pd.to_datetime("2015-01-01"), pd.to_datetime("2015-01-06")],
    ["id_0", "ch2", pd.to_datetime("2015-01-01"), pd.to_datetime("2015-01-03")],
    ["id_1", "ch1", pd.to_datetime("2015-01-05"), pd.to_datetime("2015-01-09")],
    ["id_1", "ch3", pd.to_datetime("2015-01-04"), pd.to_datetime("2015-01-09")],
    ["id_1", "ch4", pd.to_datetime("2015-01-07"), pd.to_datetime("2015-01-09")],
], columns=["ID", "Channel", "StartTime", "EndTime"])
_y_pred_ch = {
    "ch1": np.array([[pd.to_datetime("2015-01-01"), 1], [pd.to_datetime("2015-01-05 12:00"), 0]], dtype=object),
    "ch2": np.array([[pd.to_datetime("2015-01-01"), 0]], dtype=object),
    "ch3": np.array([[pd.to_datetime("2015-01-01"), 0], [pd.to_datetime("2015-01-04"), 1], [pd.to_datetime("2015-01-08"), 0]], dtype=object),
    "ch4": np.array([[pd.to_datetime("2015-01-01"), 0], [pd.to_datetime("2015-01-08"), 1]], dtype=object),
}
_result_ch = ChannelAwareFScore().score(_y_true_ch, dict(_y_pred_ch))
assert abs(_result_ch["channel_precision"] - 1.0) < 1e-9
assert abs(_result_ch["channel_recall"] - 0.75) < 1e-9
assert abs(_result_ch["channel_F0.50"] - 0.9166666666666667) < 1e-9
print("PASS: ChannelAwareFScore matches the reference implementation exactly.")


PASS: ChannelAwareFScore matches the reference implementation exactly.


## 7. ADTQC

Ported from `latency_metrics.py`. All six algorithms — see the intro on why
this one, unlike channel-aware, can be computed for the global-score
algorithms too.


In [10]:
import math

class ADTQC:
    def __init__(self, exponent: float = math.e, select_labels=None, full_range=None):
        self.exponent = exponent
        self.full_range = full_range
        if select_labels is None or len(select_labels) == 0:
            self.selected_labels = {}
        else:
            self.selected_labels = {c: np.atleast_1d(v) for c, v in select_labels.items()}

    def timing_curve(self, x, a, b):
        assert a >= pd.Timedelta(0)
        assert b >= pd.Timedelta(0)
        if (a == pd.Timedelta(0) or b == pd.Timedelta(0)) and x == pd.Timedelta(0):
            return 1
        if x <= -a or x >= b:
            return 0
        if -a < x <= pd.Timedelta(0):
            return ((x + a) / a) ** self.exponent
        if pd.Timedelta(0) < x < b:
            denom_part = x / (b - x)
            return 1. / (1. + denom_part ** self.exponent)

    def score(self, y_true: pd.DataFrame, y_pred: dict) -> dict:
        y_pred = {c: np.asarray(v) for c, v in y_pred.items()}

        min_ts = min(np.concatenate([v[..., 0] for v in y_pred.values()]))
        max_ts = max(np.concatenate([v[..., 0] for v in y_pred.values()]))
        if self.full_range is None:
            self.full_range = (min(y_true["StartTime"].min(), min_ts), max(y_true["EndTime"].max(), max_ts))

        for c in y_pred:
            if y_pred[c][0, 0] > self.full_range[0]:
                y_pred[c] = np.array([np.array([self.full_range[0], y_pred[c][0, 1]]), *y_pred[c]])
            if y_pred[c][-1, 0] < self.full_range[1]:
                y_pred[c] = np.array([*y_pred[c], np.array([self.full_range[1], y_pred[c][-1, 1]])])

        events_pred_dict = {c: convert_time_series_to_events(v) for c, v in y_pred.items()}

        filtered_y_true = y_true.copy()
        for col, val in self.selected_labels.items():
            filtered_y_true = filtered_y_true[filtered_y_true[col].isin(val)]

        unique_ids = filtered_y_true["ID"].unique()
        start_times = sorted(
            min(filtered_y_true[filtered_y_true["ID"] == aid]["StartTime"]) for aid in unique_ids
        )

        before_tps, after_tps, curve_scores = [], [], []
        for aid in unique_ids:
            gt = filtered_y_true[filtered_y_true["ID"] == aid]
            affected_channels = np.sort(gt["Channel"].unique())

            channels_intervals = {}
            for channel in affected_channels:
                c_gt = gt[gt["Channel"] == channel]
                channels_intervals[channel] = P.Interval(*[P.closed(*row) for _, row in c_gt[["StartTime", "EndTime"]].iterrows()])

            global_preds, global_gts = [], []
            for channel in affected_channels:
                if channel not in events_pred_dict:
                    continue
                matching_preds = [p for p in events_pred_dict[channel] if not (p & channels_intervals[channel]).empty]
                global_preds.extend(matching_preds)
                global_gts.append(channels_intervals[channel])
            global_preds = P.Interval(*global_preds)
            if global_preds.empty:
                continue
            global_gts = P.Interval(*global_gts)

            anomaly_length = global_gts.upper - global_gts.lower
            current_idx = start_times.index(global_gts.lower)
            previous_start = start_times[current_idx - 1] if current_idx > 0 else global_gts.lower - anomaly_length
            alpha = min(anomaly_length, global_gts.lower - previous_start)

            latency = global_preds.lower - global_gts.lower
            metric_value = self.timing_curve(latency, alpha, anomaly_length)
            curve_scores.append(metric_value)
            (before_tps if latency < pd.Timedelta(0) else after_tps).append(metric_value)

        return {
            "Nb_Before": len(before_tps),
            "Nb_After": len(after_tps),
            "AfterRate": len(after_tps) / len(curve_scores) if curve_scores else np.nan,
            "Total": np.mean(curve_scores) if curve_scores else np.nan,
        }


# --- validate against the reference's own worked example ---
_full_range_adtqc = [pd.to_datetime("8:10:10"), pd.to_datetime("8:11:24")]
_y_true_adtqc = pd.DataFrame([
    ["id_0", "ch1", pd.to_datetime("8:10:16"), pd.to_datetime("8:10:35")],
    ["id_0", "ch2", pd.to_datetime("8:10:10"), pd.to_datetime("8:10:24")],
    ["id_1", "ch3", pd.to_datetime("8:10:30"), pd.to_datetime("8:10:34")],
    ["id_1", "ch3", pd.to_datetime("8:10:40"), pd.to_datetime("8:10:45")],
    ["id_2", "ch2", pd.to_datetime("8:10:54"), pd.to_datetime("8:11:06")],
    ["id_2", "ch3", pd.to_datetime("8:10:54"), pd.to_datetime("8:11:06")],
    ["id_3", "ch1", pd.to_datetime("8:11:08"), pd.to_datetime("8:11:24")],
], columns=["ID", "Channel", "StartTime", "EndTime"])
_y_pred_adtqc = {
    "ch1": np.array([[pd.to_datetime("8:10:10"), 0], [pd.to_datetime("8:10:14"), 1], [pd.to_datetime("8:10:31"), 0], [pd.to_datetime("8:10:41"), 1]], dtype=object),
    "ch2": np.array([[pd.to_datetime("8:10:10"), 0], [pd.to_datetime("8:10:16"), 1], [pd.to_datetime("8:10:22"), 0]], dtype=object),
    "ch3": np.array([[pd.to_datetime("8:10:10"), 0], [pd.to_datetime("8:10:25"), 1], [pd.to_datetime("8:10:41"), 0]], dtype=object),
}
_result_adtqc = ADTQC(full_range=_full_range_adtqc).score(_y_true_adtqc, dict(_y_pred_adtqc))
assert _result_adtqc["Nb_Before"] == 2
assert _result_adtqc["Nb_After"] == 1
assert abs(_result_adtqc["AfterRate"] - 0.3333333333333333) < 1e-9
assert abs(_result_adtqc["Total"] - 0.4404148830816283) < 1e-9
print("PASS: ADTQC matches the reference implementation exactly.")


PASS: ADTQC matches the reference implementation exactly.


## 8. Build the per-channel prediction dicts

GlobalSTD already has genuine per-channel predictions. For the four
global-score algorithms, the same combined series gets duplicated across
every target channel's key — see the intro for why.


In [ ]:
y_pred_dicts = {}

for name in ["GlobalSTD3", "GlobalSTD5"]:
    df = globalstd_preds[name]
    y_pred_dicts[name] = {ch: np.array(list(zip(df.index, df[ch].values)), dtype=object) for ch in TARGET_CHANNELS}

for name in ["PCC", "HBOS", "iForest", "KNN"]:
    if name not in predictions_raw:
        print(f"{name}: no cached predictions found -- skipping.")
        continue
    combined = np.array(list(zip(test_df.index, predictions_raw[name])), dtype=object)
    y_pred_dicts[name] = {ch: combined for ch in TARGET_CHANNELS}

print(f"Built prediction dicts for: {list(y_pred_dicts.keys())}")


## 9. Channel-aware F-score (GlobalSTD only)

In [ ]:
full_range = (test_df.index.min(), test_df.index.max())

channel_aware_results = {}
for name in ["GlobalSTD3", "GlobalSTD5"]:
    scorer = ChannelAwareFScore(beta=BETA, full_range=full_range,
                                select_labels={"Category": ["Rare Event", "Anomaly"]})
    channel_aware_results[name] = scorer.score(y_true_with_channel, dict(y_pred_dicts[name]))
    print(f"{name}: {channel_aware_results[name]}")


## 10. ADTQC (all six)

In [ ]:
adtqc_results = {}
for name, pred_dict in y_pred_dicts.items():
    scorer = ADTQC(full_range=full_range, select_labels={"Category": ["Rare Event", "Anomaly"]})
    adtqc_results[name] = scorer.score(y_true_with_channel, dict(pred_dict))
    print(f"{name}: {adtqc_results[name]}")


## 11. Affiliation-based F-score (all six)

Unlike the other two, this one doesn't need the per-channel dict — it works
on a single combined (timestamp, is_anomaly) series and `y_true` without the
Channel column (matching the reference's own documented input shape).


In [ ]:
y_true_no_channel = y_true_with_channel.drop(columns=["Channel"]).drop_duplicates(subset=["ID", "StartTime", "EndTime"])

affiliation_results = {}
for name in ["GlobalSTD3", "GlobalSTD5"]:
    combined = globalstd_preds[name][TARGET_CHANNELS].any(axis=1).astype(np.uint8)
    y_pred_pairs = list(zip(test_df.index, combined.values))
    scorer = AffiliationScorer(betas=BETA, full_range=full_range,
                               select_labels={"Category": ["Rare Event", "Anomaly"]})
    affiliation_results[name] = scorer.score(y_true_no_channel, y_pred_pairs)
    print(f"{name}: {affiliation_results[name]}")

for name in ["PCC", "HBOS", "iForest", "KNN"]:
    if name not in predictions_raw:
        continue
    y_pred_pairs = list(zip(test_df.index, predictions_raw[name].astype(np.uint8)))
    scorer = AffiliationScorer(betas=BETA, full_range=full_range,
                               select_labels={"Category": ["Rare Event", "Anomaly"]})
    affiliation_results[name] = scorer.score(y_true_no_channel, y_pred_pairs)
    print(f"{name}: {affiliation_results[name]}")


## 12. Compare against the paper's Table 16/17

Same calibration reminder as notebooks 003/004: these are still weak
baselines by the paper's own assessment, and this comparison is a directional
check, not a target to match to three decimal places — with the added caveat
from the intro that these specific reference numbers had messier extraction
than Table 4's.


In [ ]:
print(f"{'Algorithm':<10} {'Metric':<20} {'Ours':>10} {'Paper (Table 16/17)':>20}")
print("-" * 64)
for name in ["GlobalSTD3", "GlobalSTD5", "PCC", "HBOS", "iForest", "KNN"]:
    if name not in adtqc_results:
        continue
    paper = CFG["table16"].get(name, {})
    rows = []
    if name in channel_aware_results:
        ca = channel_aware_results[name]
        rows += [
            ("channel_precision", ca["channel_precision"], paper.get("channel_precision")),
            ("channel_recall", ca["channel_recall"], paper.get("channel_recall")),
            (f"channel_F{BETA:.2f}", ca[f"channel_F{BETA:.2f}"], paper.get("channel_f0.5")),
        ]
    adtqc = adtqc_results[name]
    rows += [
        ("ADTQC after_ratio", adtqc["AfterRate"], paper.get("adtqc_after_ratio")),
        ("ADTQC score", adtqc["Total"], paper.get("adtqc_score")),
    ]
    aff = affiliation_results[name]
    rows += [
        ("AFF_precision", aff["AFF_precision"], paper.get("aff_precision")),
        ("AFF_recall", aff["AFF_recall"], paper.get("aff_recall")),
        (f"AFF_F{BETA:.2f}", aff[f"AFF_F_{BETA:.2f}"], paper.get("aff_f0.5")),
    ]
    for metric_name, ours_val, paper_val in rows:
        ours_str = f"{ours_val:.4f}" if ours_val is not None else "-"
        paper_str = f"{paper_val:.4f}" if paper_val is not None else "-"
        print(f"{name:<10} {metric_name:<20} {ours_str:>10} {paper_str:>20}")
    print()


## Next steps

- All five levels of the paper's priority hierarchy (Table 3) are now covered
  for the six models we have. That's the full evaluation picture, not just
  the headline number.
- **Windowed iForest** — still the deferred quick bonus from notebook 004,
  unsupervised so no window-labeling risk, and now it would get the full
  five-metric treatment too, not just the primary score.
- **After that:** the real return of windowing for Telemanom-ESA and
  DC-VAE-ESA — built the way notebook 003's intro described (nominal-only
  training windows, continuous-score evaluation), scored with everything
  built across notebooks 003-005 rather than starting the metrics work over.
